# Sección 1 — Gate de convergencia (wandb)

**Precondición:** entrenamiento completado (§8 del runbook sim2sim). Existe
`last.pt` en la carpeta del run de Isaac Lab.

Este gate debe cerrarse **antes de tocar el Jetson o preparar la USB**.
Si `failure_rate_mean` final ≥ 0.3 → el checkpoint no vale para sim2real,
hay que revisar los PKLs de entrenamiento y relanzar §8 del sim2sim.



## 1.0 CONFIG del gate

⚠️ Único bloque que se edita en §1. Copiar el RUN_ID del run de interés
(obtenido en §8.5 del sim2sim o desde el dashboard de wandb).

In [3]:
import os
os.environ["WANDB_DISABLE_SERVICE"] = "true"

WANDB_ENTITY  = "crisguycabs-udi"   # <- cambiar según el usuario de wandb
WANDB_PROJECT = "TRL_G1_Track"      # <- no cambiar
WANDB_RUN_ID  = "9q7w1178"          # <- editar por run

CHECKPOINT_DIR = (
    "/home/udirobotika/blea/GR00T-WholeBodyControl/"
    "logs_rl/TRL_G1_Track/manager/universal_token/"
    "all_modes/sonic_release_test-20261009_102020"
) # <- editar por run

print(f"Entity   : {WANDB_ENTITY}")
print(f"Project  : {WANDB_PROJECT}")
print(f"Run ID   : {WANDB_RUN_ID}")
print(f"Ckpt dir : {CHECKPOINT_DIR}")

Entity   : crisguycabs-udi
Project  : TRL_G1_Track
Run ID   : 9q7w1178
Ckpt dir : /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020


## 1.1 Verificar que el checkpoint existe en disco

In [4]:
import pathlib

ckpt = pathlib.Path(CHECKPOINT_DIR) / "last.pt"
print(f"Buscando: {ckpt}")
if ckpt.exists():
    size_mb = ckpt.stat().st_size / 1e6
    print(f"✅ last.pt encontrado  ({size_mb:.0f} MB)")
else:
    print("❌ last.pt NO encontrado — verificar CHECKPOINT_DIR")

Buscando: /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/last.pt
✅ last.pt encontrado  (449 MB)


## 1.2 Query de convergencia desde wandb

Descarga las métricas clave del run y muestra la tabla de evolución.
`adp_samp/failure_rate_mean` es la métrica principal — mide la fracción
de episodios en que el robot cae o termina antes del timeout.

Un run `crashed` en wandb **no indica fallo de entrenamiento**: Isaac Lab
termina abruptamente al llegar al límite de iteraciones sin graceful
shutdown. El checkpoint es válido si `last.pt` existe y tiene el tamaño
esperado (~400 MB para esta arquitectura).

In [5]:
import pathlib
import pandas as pd

# ── EDITAR ESTOS VALORES ──────────────────────────────────────────────
CHECKPOINT_DIR = (
    "/home/udirobotika/blea/GR00T-WholeBodyControl/"
    "logs_rl/TRL_G1_Track/manager/universal_token/all_modes/"
    "sonic_release_test-20261009_102020"
)
METRICS_CSV = (
    "/home/udirobotika/blea/GR00T-WholeBodyControl/"
    "logs_rl/reportes/9q7w1178/metrics.csv"
)
FPS = 50
# ──────────────────────────────────────────────────────────────────────

ckpt = pathlib.Path(CHECKPOINT_DIR) / "last.pt"
df   = pd.read_csv(METRICS_CSV)
fr   = df["adp_samp/failure_rate_mean"].dropna()
ep   = df["objective/length"].dropna()

FR_INICIO = fr.iloc[0]
FR_FINAL  = fr.iloc[-1]
FR_MIN    = fr.min()
EP_MAX    = ep.max()
TENDENCIA = FR_FINAL < fr.iloc[len(fr)//2]

checks = [
    (ckpt.exists(),                  "last.pt existe en disco"),
    (ckpt.stat().st_size > 100e6,    f"last.pt > 100 MB  ({ckpt.stat().st_size/1e6:.0f} MB)"),
    (FR_FINAL < 0.30,                f"failure_rate final < 0.30  (actual: {FR_FINAL:.4f})"),
    (TENDENCIA,                      f"failure_rate con tendencia descendente"),
]

print("=" * 50)
print("GATE §1 — CHECKLIST")
print("=" * 50)
aprobado = True
for ok, desc in checks:
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok:
        aprobado = False

print(f"\nfailure_rate  inicio : {FR_INICIO:.4f}")
print(f"failure_rate  final  : {FR_FINAL:.4f}")
print(f"failure_rate  mínimo : {FR_MIN:.4f}")
print(f"reducción            : {(1 - FR_FINAL/FR_INICIO)*100:.1f}%")
print(f"ep_length máximo     : {EP_MAX:.0f} frames  ({EP_MAX/FPS:.1f}s @ {FPS}fps)")
print()
print("✅ Gate §1 CERRADO — proceder a §2" if aprobado else "❌ Gate §1 ABIERTO")

GATE §1 — CHECKLIST
  ✅  last.pt existe en disco
  ✅  last.pt > 100 MB  (449 MB)
  ✅  failure_rate final < 0.30  (actual: 0.0218)
  ✅  failure_rate con tendencia descendente

failure_rate  inicio : 1.0000
failure_rate  final  : 0.0218
failure_rate  mínimo : 0.0218
reducción            : 97.8%
ep_length máximo     : 7739 frames  (154.8s @ 50fps)

✅ Gate §1 CERRADO — proceder a §2


## 1.3 Gate de aprobación

Criterios de interpretación de `failure_rate_mean`:

| Rango final | Significado |
|---|---|
| < 0.10 | ✅ Convergencia sólida — proceder a §2 |
| 0.10 – 0.30 | ⚠️ Convergencia parcial — aceptable si la tendencia es descendente |
| ≥ 0.30 | ❌ No convergió — revisar PKLs y relanzar §8 del sim2sim |

`error_joint_pos` en el rango 0.09–0.24 rad es esperable para Ruta 2
(política aprende la envolvente del movimiento, no réplica exacta).

In [6]:
fr_final = df["adp_samp/failure_rate_mean"].iloc[-1]
tendencia_ok = df["adp_samp/failure_rate_mean"].iloc[-1] < df["adp_samp/failure_rate_mean"].iloc[len(df)//2]

print("=" * 50)
print("GATE §1 — CHECKLIST")
print("=" * 50)

checks = [
    (ckpt.exists(),                        "last.pt existe en disco"),
    (ckpt.stat().st_size > 100e6,          "last.pt > 100 MB (no corrupto)"),
    (fr_final < 0.30,                      f"failure_rate final < 0.30  (actual: {fr_final:.4f})"),
    (tendencia_ok,                         "failure_rate con tendencia descendente"),
]

aprobado = True
for ok, descripcion in checks:
    marca = "✅" if ok else "❌"
    print(f"  {marca}  {descripcion}")
    if not ok:
        aprobado = False

print()
if aprobado:
    print("✅ Gate §1 CERRADO — proceder a §2")
    print(f"   Checkpoint: {CHECKPOINT_DIR}/last.pt")
else:
    print("❌ Gate §1 ABIERTO — no continuar hasta resolver los ❌")

GATE §1 — CHECKLIST
  ✅  last.pt existe en disco
  ✅  last.pt > 100 MB (no corrupto)
  ✅  failure_rate final < 0.30  (actual: 0.0218)
  ✅  failure_rate con tendencia descendente

✅ Gate §1 CERRADO — proceder a §2
   Checkpoint: /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/last.pt


---

# Sección 2 — Selección de repetición limpia → PKL atómico → replay MuJoCo

**Precondición:** Gate §1 cerrado.

Los PKLs en `robot_filtered_motionlib_clean/` son la única fuente de verdad
— son los aprobados que se usaron para entrenar. Esta sección selecciona
visualmente la mejor repetición, la extrae como PKL atómico en el mismo
formato motionlib, y la valida con replay en MuJoCo.

Formato motionlib: `dof` (T,29) · `root_rot` (T,4) · `root_trans_offset` (T,3) · `fps`

## 2.0 CONFIG

⚠️ Único bloque que se edita en §2.

In [7]:
import pathlib

BASE_BLEA = "/home/udirobotika/blea/GR00T-WholeBodyControl"
DIR_CLEAN = f"{BASE_BLEA}/robot_filtered_motionlib_macarena2"
FPS       = 50

pkls = sorted(pathlib.Path(DIR_CLEAN).glob("*.pkl"))
print(f"Carpeta : {DIR_CLEAN}")
print(f"PKLs    : {len(pkls)}\n")
for p in pkls:
    print(f"  {p.name}")

Carpeta : /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2
PKLs    : 6

  macarena2_cris_lento.pkl
  macarena2_cris_normal.pkl
  macarena2_cris_normal__atomic_7750_8150.pkl
  macarena2_cris_normal__atomic_7750_8150__rich.pkl
  macarena2_cris_rapido.pkl
  macarena2_cris_sostenido.pkl


## 2.1 Inspeccionar PKLs

Duración y varianza articular por grupo para cada PKL aprobado.
`std_brazo_R` alto indica repeticiones con movimiento activo del brazo
derecho — el relevante para el saludo militar.

In [10]:
import joblib
import numpy as np

# Refrescar glob aquí para excluir derivados (atomic, rich, etc.)
pkls = sorted(p for p in pathlib.Path(DIR_CLEAN).glob("*.pkl")
              if "__" not in p.name)

print(f"{'PKL':<45} {'T':>6} {'seg':>6} {'std_brazo_L':>12} {'std_brazo_R':>12}")
print("-" * 85)

resumen = []
for p in pkls:
    d     = joblib.load(p)
    key   = list(d.keys())[0]
    inner = d[key]
    if "dof" not in inner:
        print(f"{p.name:<45}  ← SKIP (keys disponibles: {list(inner.keys())})")
        continue
    dof   = np.asarray(inner["dof"])   # (T, 29)
    T     = dof.shape[0]
    std_L = dof[:, 15:22].std(0).mean()
    std_R = dof[:, 22:29].std(0).mean()
    print(f"{p.name:<45} {T:>6} {T/FPS:>6.1f} {std_L:>12.4f} {std_R:>12.4f}")
    resumen.append({"name": p.name, "path": p, "T": T, "key": key,
                    "std_L": std_L, "std_R": std_R})

print(f"\nTotal PKLs fuente: {len(resumen)}")

PKL                                                T    seg  std_brazo_L  std_brazo_R
-------------------------------------------------------------------------------------
macarena2_cris_lento.pkl                       19771  395.4       0.4745       0.4873
macarena2_cris_normal.pkl                       9353  187.1       0.4159       0.4315
macarena2_cris_rapido.pkl                       9876  197.5       0.4610       0.4874
macarena2_cris_sostenido.pkl                    9511  190.2       0.5007       0.5350

Total PKLs fuente: 4


## 2.2 Visualización de joints por PKL

Grafica los 29 joints en el tiempo para cada PKL. El eje X muestra
segundos y número de frame simultáneamente para facilitar la selección
de FRAME_START y FRAME_END en §2.3.

Imágenes guardadas en `DIR_CLEAN/plots/`.

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

out_plots = pathlib.Path(DIR_CLEAN) / "plots"
out_plots.mkdir(exist_ok=True)

GRUPOS = {
    "pierna_L (0-5)":  list(range(0, 6)),
    "pierna_R (6-11)": list(range(6, 12)),
    "cintura (12-14)": list(range(12, 15)),
    "brazo_L (15-21)": list(range(15, 22)),
    "brazo_R (22-28)": list(range(22, 29)),
}
TICK_FRAMES = 20  # <- tick cada N frames

for r in resumen:
    d   = joblib.load(r["path"])
    dof = np.asarray(d[r["key"]]["dof"])
    T   = len(dof)
    frames = np.arange(T)

    fig, axes = plt.subplots(len(GRUPOS), 1, figsize=(24, 14), sharex=True)
    fig.subplots_adjust(hspace=0.08)

    for ax, (label, idxs) in zip(axes, GRUPOS.items()):
        for i in idxs:
            ax.plot(frames, dof[:, i], linewidth=0.8, alpha=0.85, label=f"j{i}")
        ax.set_ylabel(label, fontsize=8)
        ax.grid(axis='x', linestyle=':', alpha=0.5)
        ax.legend(fontsize=6, ncol=len(idxs), loc='upper right')

    ticks = np.arange(0, T + TICK_FRAMES, TICK_FRAMES)
    ax_last = axes[-1]
    ax_last.set_xticks(ticks)
    ax_last.set_xticklabels([str(f) for f in ticks], fontsize=7, rotation=90)
    ax_last.set_xlabel("Frame", fontsize=9)
    ax_last.set_xlim(0, T - 1)

    for ax in axes:
        for tt in ticks:
            ax.axvline(tt, color='gray', linewidth=0.3, alpha=0.4)

    fig.suptitle(r["name"], fontsize=10)
    out_path = out_plots / f"{r['name'].replace('.pkl', '')}.png"
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"✅ {out_path.name}  ({T} frames)")

print(f"\nImágenes en: {out_plots}")

✅ macarena2_cris_lento.png  (19771 frames)
✅ macarena2_cris_normal.png  (9353 frames)
✅ macarena2_cris_rapido.png  (9876 frames)
✅ macarena2_cris_sostenido.png  (9511 frames)

Imágenes en: /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/plots


Zoom complementario

In [ ]:
import joblib, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE_BLEA  = "/home/udirobotika/blea/GR00T-WholeBodyControl"
PKL_ZOOM   = f"{BASE_BLEA}/robot_filtered_motionlib_macarena2/macarena2_cris_normal.pkl"
ZOOM_START = 500
ZOOM_END   = 1500
FPS        = 50

d   = joblib.load(PKL_ZOOM)
key = list(d.keys())[0]
dof = np.asarray(d[key]["dof"])[ZOOM_START:ZOOM_END]  # (1000, 29)
frames = np.arange(ZOOM_START, ZOOM_END)

GRUPOS = {
    "cintura (12-14)": list(range(12, 15)),
    "brazo_L (15-21)": list(range(15, 22)),
    "brazo_R (22-28)": list(range(22, 29)),
}

fig, axes = plt.subplots(3, 1, figsize=(24, 10), sharex=True)
fig.subplots_adjust(hspace=0.1)

for ax, (label, idxs) in zip(axes, GRUPOS.items()):
    for i in idxs:
        ax.plot(frames, dof[:, i], linewidth=1.0, label=f"j{i}")  # <- i directo
    ax.set_ylabel(label, fontsize=9)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    ax.legend(fontsize=7, ncol=len(idxs), loc='upper right')
    for tt in range(ZOOM_START, ZOOM_END, 50):
        ax.axvline(tt, color='gray', linewidth=0.4, alpha=0.5)

axes[-1].set_xlabel("Frame", fontsize=9)
axes[-1].set_xticks(range(ZOOM_START, ZOOM_END, 50))
axes[-1].set_xticklabels([str(f) for f in range(ZOOM_START, ZOOM_END, 50)],
                          fontsize=8, rotation=90)

fig.suptitle(f"ZOOM macarena2_cris_normal — frames {ZOOM_START}–{ZOOM_END}", fontsize=10)
out = f"{BASE_BLEA}/robot_filtered_motionlib_macarena2/plots/zoom_normal.png"
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.close()
print(f"✅ {out}")

## 2.3 Selección manual del segmento

⚠️ Revisar las imágenes en `DIR_CLEAN/plots/` y
editar PKL_ELEGIDO, FRAME_START y FRAME_END en la celda siguiente.

Criterios generales de selección:
- El grupo articular principal del skill muestra movimiento limpio y coherente
- Sin spikes abruptos en ningún grupo
- Piernas estables durante todo el segmento (el robot no cayó)
- El segmento corresponde a una sola repetición completa del movimiento:
  inicio en posición neutral → ejecución → retorno a posición neutral

In [12]:
# ── EDITAR ESTOS TRES VALORES ──────────────────────────────────────────
PKL_ELEGIDO = "macarena2_cris_normal.pkl"  # <- editar
FRAME_START = 7750    # <- frame inicio de la repetición limpia
FRAME_END   = 8150  # <- frame fin   de la repetición limpia
# ───────────────────────────────────────────────────────────────────────

pkl_path = pathlib.Path(DIR_CLEAN) / PKL_ELEGIDO
d   = joblib.load(pkl_path)
key = list(d.keys())[0]
dof = np.asarray(d[key]["dof"])

print(f"PKL         : {PKL_ELEGIDO}")
print(f"Motion key  : {key}")
print(f"Total frames: {len(dof)}  ({len(dof)/FPS:.1f}s)")
print(f"Segmento    : frames {FRAME_START}–{FRAME_END}  "
      f"({FRAME_END - FRAME_START} frames, "
      f"{(FRAME_END - FRAME_START)/FPS:.1f}s)")
assert FRAME_END > FRAME_START, "FRAME_END debe ser mayor que FRAME_START"
assert FRAME_END <= len(dof),   "FRAME_END fuera de rango"
print("\n✅ Segmento válido")

PKL         : macarena2_cris_normal.pkl
Motion key  : macarena2
Total frames: 9353  (187.1s)
Segmento    : frames 7750–8150  (400 frames, 8.0s)

✅ Segmento válido


In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python - << 'EOF'
import mujoco, mujoco.viewer, joblib, numpy as np, time

PKL_PATH = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal.pkl"
XML      = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic/data/robots/g1/scene_29dof_freebase.xml"
FPS      = 50

# Rango a visualizar — ajustar para explorar
START = 7750
END   = 8150  # None = hasta el final

d   = joblib.load(PKL_PATH)
key = list(d.keys())[0]
dof       = np.asarray(d[key]["dof"])
root_rot  = np.asarray(d[key]["root_rot"])   # wxyz
root_trans = np.asarray(d[key]["root_trans_offset"])

END = END or len(dof)
print(f"Key    : {key}")
print(f"Total  : {len(dof)} frames  ({len(dof)/FPS:.1f}s)")
print(f"Rango  : {START}–{END}  ({(END-START)/FPS:.1f}s)")
print("Cierra la ventana para terminar")

model = mujoco.MjModel.from_xml_path(XML)
data  = mujoco.MjData(model)

with mujoco.viewer.launch_passive(model, data) as viewer:
    for i in range(START, END):
        if i % 50 == 0:
            print(f"  frame {i}")
        data.qpos[0:3] = root_trans[i]
        data.qpos[3:7] = root_rot[i]
        data.qpos[7:]  = dof[i]
        mujoco.mj_forward(model, data)
        viewer.sync()
        time.sleep(1 / FPS)

print("Listo")
EOF

## 2.4 Construir PKL atómico

Extrae el segmento seleccionado y lo guarda como PKL atómico en
`DIR_CLEAN/` con el mismo formato motionlib.
Este PKL es la referencia de movimiento para el deploy en el Jetson.

In [14]:
import joblib

seg = slice(FRAME_START, FRAME_END)

atomic_key  = key  # conserva el mismo motion key
atomic_data = {
    atomic_key: {
        "dof":               np.asarray(d[key]["dof"])[seg].astype(np.float32),
        "root_rot":          np.asarray(d[key]["root_rot"])[seg].astype(np.float32),
        "root_trans_offset": np.asarray(d[key]["root_trans_offset"])[seg].astype(np.float32),
        "fps":               FPS,
    }
}

T_atomic = atomic_data[atomic_key]["dof"].shape[0]
nombre_atomic = f"{PKL_ELEGIDO.replace('.pkl','')}__atomic_{FRAME_START}_{FRAME_END}.pkl"
out_atomic = pathlib.Path(DIR_CLEAN) / nombre_atomic

joblib.dump(atomic_data, out_atomic, compress=True)

print(f"PKL atómico guardado : {out_atomic.name}")
print(f"Frames               : {T_atomic}  ({T_atomic/FPS:.1f}s @ {FPS}fps)")
print(f"dof shape            : {atomic_data[atomic_key]['dof'].shape}")
print(f"root_rot shape       : {atomic_data[atomic_key]['root_rot'].shape}")
print(f"root_trans shape     : {atomic_data[atomic_key]['root_trans_offset'].shape}")

PKL atómico guardado : macarena2_cris_normal__atomic_7750_8150.pkl
Frames               : 400  (8.0s @ 50fps)
dof shape            : (400, 29)
root_rot shape       : (400, 4)
root_trans shape     : (400, 3)


## 2.5 Replay MuJoCo del PKL atómico

⚠️ El visor MuJoCo no puede lanzarse desde VSCode — ejecutar en terminal.
Copiar el bloque de abajo y pegarlo directo en la terminal con el
env `.venv_teleop` activo.

Root fijo en origen — flotación leve de pies es normal.
Lo que importa es la forma del movimiento articular.

⚠️ Editar PKL_PATH y KEY con los valores de §2.3 antes de ejecutar.

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python - << 'EOF'
import mujoco, mujoco.viewer, joblib, numpy as np, time

PKL_PATH = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150.pkl"
XML      = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic/data/robots/g1/scene_29dof_freebase.xml"
FPS      = 50
REPS     = 5

d   = joblib.load(PKL_PATH)
KEY = list(d.keys())[0]
dof = np.asarray(d[KEY]["dof"])

print(f"Key           : {KEY}")
print(f"Frames        : {len(dof)}  ({len(dof)/FPS:.1f}s)")
print(f"Repeticiones  : {REPS}")
print("Cerrar ventana para terminar")

model = mujoco.MjModel.from_xml_path(XML)
data  = mujoco.MjData(model)

with mujoco.viewer.launch_passive(model, data) as viewer:
    for rep in range(REPS):
        print(f"Rep {rep+1}/{REPS}...")
        for frame in dof:
            data.qpos[7:] = frame
            mujoco.mj_forward(model, data)
            viewer.sync()
            time.sleep(1 / FPS)

print("Replay completado")
EOF

## 2.6 Gate visual §2

- [ ] El movimiento es reconocible
- [ ] Sin teleportaciones ni spikes articulares
- [ ] Piernas estables durante todo el segmento
- [ ] Duración coherente con una repetición real

✅ Gate §2 cerrado. Anotar para las secciones siguientes:
- `PKL_ELEGIDO`  = ...
- `FRAME_START`  = ...
- `FRAME_END`    = ...
- `nombre_atomic`= ...

# Sección 3 — Enriquecer PKL atómico con cinemática completa

**Precondición:** PKL atómico generado en §2.4.

El PKL atómico tiene formato motionlib (`dof`, `root_rot`, `root_trans_offset`,
`fps`). `convert_motions.py` necesita 6 arrays. Esta sección los produce
vía FK en MuJoCo + diferencias finitas:

| Campo a producir | Fuente |
|---|---|
| `joint_pos`      (T,29) | directo desde `dof` |
| `joint_vel`      (T,29) | gradiente de `dof` × fps |
| `body_pos_w`    (T,14,3) | FK: `data.xpos` para los 14 bodies |
| `body_quat_w`   (T,14,4) | FK: `data.xquat` para los 14 bodies |
| `body_lin_vel_w`(T,14,3) | gradiente de `body_pos_w` × fps |
| `body_ang_vel_w`(T,14,3) | derivada de quaterniones × fps |

⚠️ Esta sección corre en el notebook con kernel isaaclab.
MuJoCo está disponible vía `.venv_teleop` — importarlo con subprocess.
Usamos `importlib` para acceder al venv correcto.



## 3.0 CONFIG

⚠️ Si PKL_ELEGIDO, FRAME_START, FRAME_END ya están definidos en §2
(misma sesión de kernel), esta celda no necesita edición.

In [16]:
import pathlib, sys

# Rutas — consistentes con §2
BASE_BLEA  = "/home/udirobotika/blea/GR00T-WholeBodyControl"
DIR_CLEAN  = f"{BASE_BLEA}/robot_filtered_motionlib_macarena2"
FPS        = 50
XML        = f"{BASE_BLEA}/gear_sonic/data/robots/g1/scene_29dof_freebase.xml"

# PKL atómico de §2.4 — editar si es nueva sesión
ATOMIC_PKL = str(out_atomic.name)   # ya definido en §2; editar si kernel fue reiniciado

# ── EDITAR si el kernel fue reiniciado ─────────────────────────────────
# ATOMIC_PKL = (
#     "/home/udirobotika/blea/GR00T-WholeBodyControl/"
#     "robot_filtered_motionlib_macarena2/"
#     "macarena2_cris_normal__atomic_7750_8150.pkl"
# )
# ───────────────────────────────────────────────────────────────────────

# Los 14 body IDs de MuJoCo correspondientes a los body_names del training
BODY_IDS = [1, 3, 5, 7, 13, 15, 17, 24, 26, 28, 31, 34, 36, 39]
BODY_NAMES = [
    "pelvis", "left_hip_roll_link", "left_knee_link", "left_ankle_roll_link",
    "right_hip_roll_link", "right_knee_link", "right_ankle_roll_link",
    "torso_link", "left_shoulder_roll_link", "left_elbow_link",
    "left_wrist_yaw_link", "right_shoulder_roll_link",
    "right_elbow_link", "right_wrist_yaw_link",
]

print(f"PKL atómico : {ATOMIC_PKL}")
print(f"XML         : {XML}")
print(f"Bodies      : {len(BODY_IDS)}")

PKL atómico : macarena2_cris_normal__atomic_7750_8150.pkl
XML         : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic/data/robots/g1/scene_29dof_freebase.xml
Bodies      : 14


## 3.1 Cargar PKL atómico y verificar campos

In [17]:
ATOMIC_PKL

'macarena2_cris_normal__atomic_7750_8150.pkl'

In [18]:
import joblib
import numpy as np

# d   = joblib.load(ATOMIC_PKL)
d   = joblib.load(f"{BASE_BLEA}/robot_filtered_motionlib_macarena2/{ATOMIC_PKL}")
key = list(d.keys())[0]
seg = d[key]

dof               = np.asarray(seg["dof"],               dtype=np.float32)  # (T,29)
root_rot          = np.asarray(seg["root_rot"],           dtype=np.float32)  # (T,4) wxyz
root_trans_offset = np.asarray(seg["root_trans_offset"],  dtype=np.float32)  # (T,3)
T = dof.shape[0]

print(f"Key              : {key}")
print(f"Frames           : {T}  ({T/FPS:.1f}s @ {FPS}fps)")
print(f"dof              : {dof.shape}")
print(f"root_rot         : {root_rot.shape}")
print(f"root_trans_offset: {root_trans_offset.shape}")
assert dof.shape == (T, 29), "dof debe ser (T,29)"
assert root_rot.shape == (T, 4), "root_rot debe ser (T,4)"
print("\n✅ PKL atómico cargado")

Key              : macarena2
Frames           : 400  (8.0s @ 50fps)
dof              : (400, 29)
root_rot         : (400, 4)
root_trans_offset: (400, 3)

✅ PKL atómico cargado


In [19]:
# ── Permutación IL ↔ MJ ────────────────────────────────────────────────
# El binary C++ espera joints en Isaac Lab order y aplica IL→MJ internamente.
# El PKL atómico tiene dof en MuJoCo order → hay que convertir antes del rich PKL.
# La FK (§3.2) usa el archivo PKL directamente en MJ order → no se ve afectada.

IL_TO_MJ = np.array([0,3,6,9,13,17,1,4,7,10,14,18,2,5,8,11,15,19,21,23,25,27,12,16,20,22,24,26,28])
MJ_TO_IL = np.argsort(IL_TO_MJ)

dof_il = dof[:, MJ_TO_IL]   # (T,29) — IL order para los CSVs

# Verificación: round-trip debe recuperar el dof original
assert np.allclose(dof_il[:, IL_TO_MJ], dof), "Round-trip IL↔MJ falló — revisar permutación"
print(f"IL_TO_MJ : {IL_TO_MJ}")
print(f"MJ_TO_IL : {MJ_TO_IL}")
print(f"dof_il   : {dof_il.shape}  (IL order — para joint_pos en rich PKL)")
print(f"dof      : {dof.shape}     (MJ order — para FK en §3.2)")
print("\n✅ Permutación IL↔MJ aplicada y verificada")

IL_TO_MJ : [ 0  3  6  9 13 17  1  4  7 10 14 18  2  5  8 11 15 19 21 23 25 27 12 16
 20 22 24 26 28]
MJ_TO_IL : [ 0  6 12  1  7 13  2  8 14  3  9 15 22  4 10 16 23  5 11 17 24 18 25 19
 26 20 27 21 28]
dof_il   : (400, 29)  (IL order — para joint_pos en rich PKL)
dof      : (400, 29)     (MJ order — para FK en §3.2)

✅ Permutación IL↔MJ aplicada y verificada


## 3.2 FK en MuJoCo → body_pos_w y body_quat_w

MuJoCo corre en `.venv_teleop` (Python 3.10) — incompatible con el kernel
isaaclab (Python 3.11). El FK se ejecuta como subproceso: guarda los
resultados en `robot_filtered_motionlib_clean/fk_output.npz` y los carga
en el notebook.

⚠️ Copiar el bloque bash de la celda siguiente y pegarlo en terminal.

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python - << 'EOF'
import mujoco, joblib, numpy as np

ATOMIC_PKL = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150.pkl"
XML        = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic/data/robots/g1/scene_29dof_freebase.xml"
OUT_NPZ    = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/fk_output.npz"
BODY_IDS   = [1, 3, 5, 7, 13, 15, 17, 24, 26, 28, 31, 34, 36, 39]

d   = joblib.load(ATOMIC_PKL)
key = list(d.keys())[0]
seg = d[key]

dof               = np.asarray(seg["dof"],               dtype=np.float32)
root_rot          = np.asarray(seg["root_rot"],           dtype=np.float32)
root_trans_offset = np.asarray(seg["root_trans_offset"],  dtype=np.float32)
T = dof.shape[0]

model = mujoco.MjModel.from_xml_path(XML)
data  = mujoco.MjData(model)

# Verificar body IDs
BODY_NAMES = [
    "pelvis", "left_hip_roll_link", "left_knee_link", "left_ankle_roll_link",
    "right_hip_roll_link", "right_knee_link", "right_ankle_roll_link",
    "torso_link", "left_shoulder_roll_link", "left_elbow_link",
    "left_wrist_yaw_link", "right_shoulder_roll_link",
    "right_elbow_link", "right_wrist_yaw_link",
]
print("Verificación de body IDs:")
for i, (bid, bname) in enumerate(zip(BODY_IDS, BODY_NAMES)):
    mj_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, bid)
    ok = "✅" if mj_name == bname else "❌"
    print(f"  {ok} [{i:02d}] MuJoCo[{bid:02d}] {mj_name}")

# Loop FK
body_pos_w  = np.zeros((T, 14, 3), dtype=np.float32)
body_quat_w = np.zeros((T, 14, 4), dtype=np.float32)

for t in range(T):
    data.qpos[0:3] = root_trans_offset[t]
    data.qpos[3:7] = root_rot[t]
    data.qpos[7:]  = dof[t]
    mujoco.mj_forward(model, data)
    body_pos_w[t]  = data.xpos[BODY_IDS]
    body_quat_w[t] = data.xquat[BODY_IDS]
    if t % 50 == 0:
        print(f"  frame {t:4d}/{T}")

np.savez(OUT_NPZ, body_pos_w=body_pos_w, body_quat_w=body_quat_w)
print(f"\n✅ FK guardado en {OUT_NPZ}")
print(f"body_pos_w  : {body_pos_w.shape}")
print(f"body_quat_w : {body_quat_w.shape}")
EOF

In [21]:
import numpy as np

OUT_NPZ = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/fk_output.npz"
fk = np.load(OUT_NPZ)

body_pos_w  = fk["body_pos_w"]
body_quat_w = fk["body_quat_w"]

print(f"✅ FK cargado desde {OUT_NPZ}")
print(f"body_pos_w  : {body_pos_w.shape}")
print(f"body_quat_w : {body_quat_w.shape}")

✅ FK cargado desde /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/fk_output.npz
body_pos_w  : (400, 14, 3)
body_quat_w : (400, 14, 4)


## 3.3 Velocidades por diferencias finitas

In [22]:
dt = 1.0 / FPS

# joint_vel: gradiente de dof_il (IL order — consistente con joint_pos)
joint_vel = np.gradient(dof_il, dt, axis=0).astype(np.float32)

# body_lin_vel_w: gradiente de posiciones
body_lin_vel_w = np.gradient(body_pos_w, dt, axis=0).astype(np.float32)

# body_ang_vel_w: derivada desde quaterniones (mundo)
body_ang_vel_w = np.zeros((T, 14, 3), dtype=np.float32)
q = body_quat_w  # (T,14,4) wxyz

for t in range(T):
    if t == 0:
        dq = (q[1] - q[0]) / dt
    elif t == T - 1:
        dq = (q[-1] - q[-2]) / dt
    else:
        dq = (q[t+1] - q[t-1]) / (2 * dt)

    qw = q[t, :, 0];  qx = q[t, :, 1]
    qy = q[t, :, 2];  qz = q[t, :, 3]
    dw = dq[:, 0];    dx = dq[:, 1]
    dy = dq[:, 2];    dz = dq[:, 3]

    body_ang_vel_w[t, :, 0] = 2 * ( qw*dx - dw*qx + dy*qz - dz*qy)
    body_ang_vel_w[t, :, 1] = 2 * ( qw*dy - dw*qy + dz*qx - dx*qz)
    body_ang_vel_w[t, :, 2] = 2 * ( qw*dz - dw*qz + dx*qy - dy*qx)

print("✅ Velocidades calculadas")
print(f"joint_vel       : {joint_vel.shape}  max={np.abs(joint_vel).max():.3f} rad/s")
print(f"body_lin_vel_w  : {body_lin_vel_w.shape}  max={np.abs(body_lin_vel_w).max():.3f} m/s")
print(f"body_ang_vel_w  : {body_ang_vel_w.shape}  max={np.abs(body_ang_vel_w).max():.3f} rad/s")

✅ Velocidades calculadas
joint_vel       : (400, 29)  max=7.305 rad/s
body_lin_vel_w  : (400, 14, 3)  max=1.636 m/s
body_ang_vel_w  : (400, 14, 3)  max=8.633 rad/s


## 3.4 Empacar PKL enriquecido y guardar

In [25]:
nombre_enriquecido = pathlib.Path(ATOMIC_PKL).stem + "__rich.pkl"
out_rich = pathlib.Path(DIR_CLEAN) / nombre_enriquecido

rich_data = {
    key: {
        "joint_pos":       dof_il,          # ← IL order (fix Bug 5)
        "joint_vel":       joint_vel,
        "body_pos_w":      body_pos_w,
        "body_quat_w":     body_quat_w,
        "body_lin_vel_w":  body_lin_vel_w,
        "body_ang_vel_w":  body_ang_vel_w,
    }
}

joblib.dump(rich_data, out_rich, compress=True)

# Verificación: joint_pos debe estar en IL order
loaded = joblib.load(out_rich)
jp = np.asarray(loaded[key]["joint_pos"])
assert np.allclose(jp, dof_il), "joint_pos guardado no coincide con dof_il"

print(f"PKL enriquecido : {out_rich.name}")
print(f"Campos          :")
for k, v in rich_data[key].items():
    print(f"  {k:<20} {np.asarray(v).shape}")
print(f"\n✅ Guardado en {out_rich}  (joint_pos en IL order ✓)")

PKL enriquecido : macarena2_cris_normal__atomic_7750_8150__rich.pkl
Campos          :
  joint_pos            (400, 29)
  joint_vel            (400, 29)
  body_pos_w           (400, 14, 3)
  body_quat_w          (400, 14, 4)
  body_lin_vel_w       (400, 14, 3)
  body_ang_vel_w       (400, 14, 3)

✅ Guardado en /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl  (joint_pos en IL order ✓)


## 3.5 Gate §3

- [ ] `body_pos_w` max < 3.0 m  (el robot no salió del origen)
- [ ] `joint_vel` max < 30 rad/s (sin spikes de velocidad)
- [ ] `body_lin_vel_w` max < 5.0 m/s (sin teleportaciones)
- [ ] `body_ang_vel_w` max < 50 rad/s (quaterniones sin discontinuidades)

✅ Gate §3 cerrado → pasar a §4.
Anotar: `out_rich` = ruta del PKL enriquecido.

In [26]:
checks = [
    (np.abs(body_pos_w).max() < 3.0,
     f"body_pos_w max < 3.0 m       (actual: {np.abs(body_pos_w).max():.3f})"),
    (np.abs(joint_vel).max() < 30.0,
     f"joint_vel max < 30 rad/s     (actual: {np.abs(joint_vel).max():.3f})"),
    (np.abs(body_lin_vel_w).max() < 5.0,
     f"body_lin_vel_w max < 5.0 m/s (actual: {np.abs(body_lin_vel_w).max():.3f})"),
    (np.abs(body_ang_vel_w).max() < 50.0,
     f"body_ang_vel_w max < 50 rad/s(actual: {np.abs(body_ang_vel_w).max():.3f})"),
]

print("=" * 55)
print("GATE §3 — CHECKLIST")
print("=" * 55)
aprobado = True
for ok, desc in checks:
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok:
        aprobado = False

print()
print(f"{'✅ Gate §3 CERRADO — proceder a §4' if aprobado else '❌ Gate §3 ABIERTO — revisar valores fuera de rango'}")
if aprobado:
    print(f"\n   PKL enriquecido: {out_rich}")

GATE §3 — CHECKLIST
  ✅  body_pos_w max < 3.0 m       (actual: 1.278)
  ✅  joint_vel max < 30 rad/s     (actual: 7.305)
  ✅  body_lin_vel_w max < 5.0 m/s (actual: 1.636)
  ✅  body_ang_vel_w max < 50 rad/s(actual: 8.633)

✅ Gate §3 CERRADO — proceder a §4

   PKL enriquecido: /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl


# Sección 4 — Convertir PKL enriquecido a reference CSV

**Precondición:** Gate §3 cerrado. PKL `__rich.pkl` disponible.

`convert_motions.py` lee el PKL enriquecido y produce el directorio
de CSVs que consume el binario `g1_deploy_onnx_ref` en el Jetson.

Estructura de salida:

In [ ]:
gear_sonic_deploy/reference/<skill>/
<motion_key>/
joint_pos.csv joint_vel.csv
body_pos.csv body_quat.csv
body_lin_vel.csv body_ang_vel.csv
metadata.txt info.txt
motion_summary.txt

El `--motion-data` del deploy apunta a `reference/<skill>/`.



## 4.0 CONFIG

⚠️ Editar SKILL_NAME si es una nueva sesión de kernel.

In [27]:
import pathlib

# ── EDITAR si el kernel fue reiniciado ─────────────────────────────────
RICH_PKL   = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl"
SKILL_NAME = "macarena2"   # <- nombre del subdirectorio en reference/
# ───────────────────────────────────────────────────────────────────────

BASE_BLEA      = "/home/udirobotika/blea/GR00T-WholeBodyControl"
CONVERT_SCRIPT = f"{BASE_BLEA}/gear_sonic_deploy/reference/convert_motions.py"
OUT_REF_DIR    = f"{BASE_BLEA}/gear_sonic_deploy/reference/{SKILL_NAME}"

print(f"PKL enriquecido : {RICH_PKL}")
print(f"Script          : {CONVERT_SCRIPT}")
print(f"Directorio ref  : {OUT_REF_DIR}")

assert pathlib.Path(RICH_PKL).exists(),      "❌ RICH_PKL no encontrado"
assert pathlib.Path(CONVERT_SCRIPT).exists(),"❌ convert_motions.py no encontrado"
print("\n✅ Paths verificados")

PKL enriquecido : /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl
Script          : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/convert_motions.py
Directorio ref  : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2

✅ Paths verificados


## 4.1 Ejecutar convert_motions.py

In [28]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, CONVERT_SCRIPT, RICH_PKL, OUT_REF_DIR],
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("❌ convert_motions.py falló")

G1 Motion Data Converter
Converting motion data from: /home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl
Output directory structure: /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2/
✓ Successfully loaded with joblib

Found 1 motion sequences:
  - macarena2

Processing: macarena2
Creating individual folder for this motion: /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2/macarena2
  Timesteps: 400, Joints: 29, Body parts: 14
  ✓ Saved 7 files for macarena2 (joints + full body kinematics)

✓ Successfully converted 1/1 motions
Output files saved to: /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2/

✓ Conversion completed successfully!

Extracted data for each motion:
- Joint positions & velocities (29 joints)
- Body positions in world coordinates (14 body parts)
- Body orientations (quaternions)
- Body li

## 4.2 Verificar estructura de salida

In [29]:
import pathlib

ref_dir = pathlib.Path(OUT_REF_DIR)
EXPECTED_FILES = [
    "joint_pos.csv", "joint_vel.csv",
    "body_pos.csv",  "body_quat.csv",
    "body_lin_vel.csv", "body_ang_vel.csv",
    "metadata.txt",  "info.txt",
]

print(f"Directorio base : {ref_dir}")
print(f"motion_summary  : {'✅' if (ref_dir/'motion_summary.txt').exists() else '❌'}\n")

# Buscar subdirectorios de motions
motion_dirs = [d for d in ref_dir.iterdir() if d.is_dir()]
print(f"Motions encontrados: {len(motion_dirs)}\n")

all_ok = True
for mdir in motion_dirs:
    print(f"  {mdir.name}/")
    for fname in EXPECTED_FILES:
        fpath = mdir / fname
        if fpath.exists():
            size_kb = fpath.stat().st_size / 1024
            print(f"    ✅ {fname:<22} {size_kb:6.1f} KB")
        else:
            print(f"    ❌ {fname}  — FALTANTE")
            all_ok = False

Directorio base : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2
motion_summary  : ✅

Motions encontrados: 1

  macarena2/
    ✅ joint_pos.csv           107.6 KB
    ✅ joint_vel.csv           107.9 KB
    ✅ body_pos.csv            153.0 KB
    ✅ body_quat.csv           203.8 KB
    ✅ body_lin_vel.csv        156.0 KB
    ✅ body_ang_vel.csv        157.2 KB
    ✅ metadata.txt              0.3 KB
    ✅ info.txt                  1.0 KB


## 4.3 Gate §4

In [30]:
import numpy as np

T_EXPECTED = T  # 400 frames del PKL atómico actual

motion_dir = motion_dirs[0]

def csv_shape(path):
    import csv
    with open(path) as f:
        rows = list(csv.reader(f))
    return (len(rows) - 1, len(rows[0])) if rows else (0, 0)

jp_rows, jp_cols = csv_shape(motion_dir / "joint_pos.csv")
bp_rows, bp_cols = csv_shape(motion_dir / "body_pos.csv")

checks = [
    (all_ok,                          "Todos los archivos CSV presentes"),
    (jp_rows == T_EXPECTED,           f"joint_pos.csv filas == {T_EXPECTED}       (actual: {jp_rows})"),
    (jp_cols == 29,                   f"joint_pos.csv cols == 29         (actual: {jp_cols})"),
    (bp_rows == T_EXPECTED,           f"body_pos.csv  filas == {T_EXPECTED}       (actual: {bp_rows})"),
    (bp_cols == 14*3,                 f"body_pos.csv  cols == 42 (14×3)  (actual: {bp_cols})"),
]

print("=" * 55)
print("GATE §4 — CHECKLIST")
print("=" * 55)
aprobado = True
for ok, desc in checks:
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok:
        aprobado = False

print()
if aprobado:
    print("✅ Gate §4 CERRADO — proceder a §5")
    print(f"\n   motion-data para deploy : {OUT_REF_DIR}/")
else:
    print("❌ Gate §4 ABIERTO — revisar errores arriba")

GATE §4 — CHECKLIST
  ✅  Todos los archivos CSV presentes
  ✅  joint_pos.csv filas == 400       (actual: 400)
  ✅  joint_pos.csv cols == 29         (actual: 29)
  ✅  body_pos.csv  filas == 400       (actual: 400)
  ✅  body_pos.csv  cols == 42 (14×3)  (actual: 42)

✅ Gate §4 CERRADO — proceder a §5

   motion-data para deploy : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/reference/macarena2/


---

# Sección 5 — Export `last.pt` → ONNX

**Precondición:** Gate §4 cerrado. Alienware con Isaac Lab disponible. Robot **no necesario**.

`eval_agent_trl.py +export_onnx_only=true` carga el checkpoint, construye el grafo
de la política finetuneada y exporta dos modelos ONNX:

- `model_step_001000_encoder.onnx` — codifica la observación a tokens latentes
- `model_step_001000_decoder.onnx` — decodifica latentes a comandos articulares

Los archivos se generan en `logs_rl/…/exported/` y luego se copian a
`gear_sonic_deploy/policy/saludo_militar/`.

⚠️ Isaac Sim tarda 15–20 min en arrancar. El export en sí toma ~2 min adicionales
una vez levantado. No cerrar la terminal durante este proceso.

## 5.0 CONFIG

⚠️ Único bloque que se edita en §5.

In [33]:
import pathlib

BASE_BLEA   = "/home/udirobotika/blea/GR00T-WholeBodyControl"
ISAACLAB_SH = "/home/udirobotika/IsaacLab/isaaclab.sh"

CHECKPOINT_DIR = (
    f"{BASE_BLEA}/logs_rl/TRL_G1_Track/manager/universal_token/"
    "all_modes/sonic_release_test-20261009_102020"
)
STEP        = "001000"               # <- número de iteración del checkpoint
SKILL_NAME  = "macarena2"       # <- nombre del subdirectorio en policy/

EXPORTED_DIR   = f"{CHECKPOINT_DIR}/exported"
POLICY_DEST    = f"{BASE_BLEA}/gear_sonic_deploy/policy/{SKILL_NAME}"

ENCODER_NAME   = f"model_step_{STEP}_encoder.onnx"
DECODER_NAME   = f"model_step_{STEP}_decoder.onnx"

print(f"Checkpoint dir : {CHECKPOINT_DIR}")
print(f"Exported dir   : {EXPORTED_DIR}")
print(f"Policy dest    : {POLICY_DEST}")
print(f"Encoder        : {ENCODER_NAME}")
print(f"Decoder        : {DECODER_NAME}")

Checkpoint dir : /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020
Exported dir   : /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/exported
Policy dest    : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2
Encoder        : model_step_001000_encoder.onnx
Decoder        : model_step_001000_decoder.onnx


## 5.1 Verificar `last.pt` en disco

In [34]:
ckpt = pathlib.Path(CHECKPOINT_DIR) / "last.pt"

print(f"Buscando: {ckpt}")
if ckpt.exists():
    size_mb = ckpt.stat().st_size / 1e6
    print(f"✅ last.pt encontrado  ({size_mb:.0f} MB)")
else:
    print("❌ last.pt NO encontrado — verificar CHECKPOINT_DIR")
    raise FileNotFoundError(str(ckpt))

assert ckpt.stat().st_size > 100e6, "❌ last.pt sospechosamente pequeño — posible corrupción"
print("✅ Tamaño OK")

Buscando: /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/last.pt
✅ last.pt encontrado  (449 MB)
✅ Tamaño OK


## 5.2 Ejecutar export ONNX

⚠️ Este paso lanza Isaac Sim — ejecutar en **terminal**, no en el notebook.
Copiar el bloque de abajo y pegarlo en una terminal nueva con `conda activate isaaclab`.

El proceso tarda 15–20 min (arranque de Sim) + ~2 min (export en sí).
Al finalizar imprime las rutas de los .onnx generados.



Se mueven los archivos atomicos para evitar problemas con el siguiente bloque

In [ ]:
mkdir -p ~/blea/GR00T-WholeBodyControl/motionlib_derived

mv ~/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150.pkl \
   ~/blea/GR00T-WholeBodyControl/motionlib_derived/

mv ~/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/macarena2_cris_normal__atomic_7750_8150__rich.pkl \
   ~/blea/GR00T-WholeBodyControl/motionlib_derived/

echo "--- robot_filtered_motionlib_macarena2 ---"
find ~/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2/ -name "*.pkl" | sort

echo ""
echo "--- motionlib_derived ---"
find ~/blea/GR00T-WholeBodyControl/motionlib_derived/ -name "*.pkl" | sort

In [ ]:
conda activate isaaclab
cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/eval_agent_trl.py \
  +checkpoint=logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/last.pt \
  +num_envs=1 \
  +headless=true \
  +export_onnx_only=true

Salida esperada al terminar:

[INFO] Exported encoder → .../exported/model_step_001000_encoder.onnx

[INFO] Exported decoder → .../exported/model_step_001000_decoder.onnx


Una vez que la terminal confirme los archivos generados, continuar con §5.3.

## 5.3 Verificar ONNX exportados

In [35]:
exported = pathlib.Path(EXPORTED_DIR)

enc = exported / ENCODER_NAME
dec = exported / DECODER_NAME

print(f"Buscando en: {exported}\n")

checks_onnx = [
    (enc.exists(), f"encoder existe : {ENCODER_NAME}"),
    (dec.exists(), f"decoder existe : {DECODER_NAME}"),
]

all_found = True
for ok, desc in checks_onnx:
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok:
        all_found = False

if all_found:
    print(f"\n  encoder : {enc.stat().st_size / 1e6:.1f} MB")
    print(f"  decoder : {dec.stat().st_size / 1e6:.1f} MB")
    print("\n✅ ONNX exportados verificados — proceder a §5.4")
else:
    raise FileNotFoundError("❌ Uno o más .onnx no encontrados — verificar salida de §5.2")

Buscando en: /home/udirobotika/blea/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20261009_102020/exported

  ✅  encoder existe : model_step_001000_encoder.onnx
  ✅  decoder existe : model_step_001000_decoder.onnx

  encoder : 50.2 MB
  decoder : 40.9 MB

✅ ONNX exportados verificados — proceder a §5.4


## 5.4 Copiar ONNX a `gear_sonic_deploy/policy/<skill>/`

In [36]:
import shutil

dest = pathlib.Path(POLICY_DEST)
dest.mkdir(parents=True, exist_ok=True)

for src in [enc, dec]:
    dst = dest / src.name
    shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / 1e6
    print(f"✅ {src.name}  →  {dst}  ({size_mb:.1f} MB)")

print(f"\n✅ Archivos en {POLICY_DEST}")

✅ model_step_001000_encoder.onnx  →  /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2/model_step_001000_encoder.onnx  (50.2 MB)
✅ model_step_001000_decoder.onnx  →  /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2/model_step_001000_decoder.onnx  (40.9 MB)

✅ Archivos en /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2


## 5.5 Gate §5

In [38]:
enc_dest = pathlib.Path(POLICY_DEST) / ENCODER_NAME
dec_dest = pathlib.Path(POLICY_DEST) / DECODER_NAME

checks = [
    (enc_dest.exists() and enc_dest.stat().st_size > 1e6,
     f"encoder en policy/macarena2/ ({enc_dest.stat().st_size/1e6:.1f} MB)" if enc_dest.exists() else "encoder NO encontrado"),
    (dec_dest.exists() and dec_dest.stat().st_size > 1e6,
     f"decoder en policy/macarena2/ ({dec_dest.stat().st_size/1e6:.1f} MB)" if dec_dest.exists() else "decoder NO encontrado"),
    (enc_dest.exists() and enc_dest.stat().st_size != dec_dest.stat().st_size,
     "encoder ≠ decoder en tamaño (no intercambiados)") if enc_dest.exists() and dec_dest.exists() else (False, "no se puede comparar tamaños"),
]

print("=" * 55)
print("GATE §5 — CHECKLIST")
print("=" * 55)
aprobado = True
for ok, desc in checks:
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok:
        aprobado = False

print()
if aprobado:
    print("✅ Gate §5 CERRADO — proceder a §6")
    print(f"\n   Checkpoint step : {STEP}")
    print(f"   Encoder         : {POLICY_DEST}/{ENCODER_NAME}")
    print(f"   Decoder         : {POLICY_DEST}/{DECODER_NAME}")
else:
    print("❌ Gate §5 ABIERTO — resolver ❌ antes de continuar")

GATE §5 — CHECKLIST
  ✅  encoder en policy/macarena2/ (50.2 MB)
  ✅  decoder en policy/macarena2/ (40.9 MB)
  ✅  encoder ≠ decoder en tamaño (no intercambiados)

✅ Gate §5 CERRADO — proceder a §6

   Checkpoint step : 001000
   Encoder         : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2/model_step_001000_encoder.onnx
   Decoder         : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/macarena2/model_step_001000_decoder.onnx


---

# Sección 6 — Empacar USB

**Precondición:** Gate §5 cerrado. USB conectada al Alienware.

Copia a la USB todo lo que necesita el Jetson: los ONNX de la política,
los CSVs de referencia, el planner, el config de observaciones, y el
código fuente C++ para compilar en aarch64.

La estructura en la USB replica `gear_sonic_deploy/` — el Jetson la monta
y reproduce la misma jerarquía.

⚠️ Verificar que la USB esté montada antes de correr §6.1.

## 6.0 CONFIG

⚠️ Único bloque que se edita en §6.
Ajustar USB_MOUNT al punto de montaje real (verificar con `lsblk` o `df -h`).

In [39]:
import pathlib, shutil

BASE_BLEA   = "/home/udirobotika/blea/GR00T-WholeBodyControl"
DEPLOY_DIR  = f"{BASE_BLEA}/gear_sonic_deploy"
SKILL_NAME  = "macarena2"
STEP        = "001000"

# ── EDITAR: punto de montaje ────────────────────────────────────────────
# DEPLOY REAL  → USB_MOUNT = "/media/udirobotika/BLEA_DEPLOY"
# MOCK         → USB_MOUNT = "/home/udirobotika/usb_mock"
USB_MOUNT = "/run/media/udirobotika/A81E-67F1"
MOCK        = False   # <- cambiar a False en deploy real
# ───────────────────────────────────────────────────────────────────────

USB_DEPLOY  = f"{USB_MOUNT}/gear_sonic_deploy"

print(f"Fuente  : {DEPLOY_DIR}")
print(f"USB     : {USB_DEPLOY}")
print(f"Modo    : {'⚠️  MOCK (carpeta local)' if MOCK else '🔌 REAL (USB física)'}")

usb = pathlib.Path(USB_MOUNT)
if MOCK:
    usb.mkdir(parents=True, exist_ok=True)
else:
    if not usb.exists():
        raise RuntimeError(f"❌ USB no montada en {USB_MOUNT} — verificar con: lsblk")

print(f"\n✅ USB encontrada  ({usb})")

Fuente  : /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy
USB     : /run/media/udirobotika/A81E-67F1/gear_sonic_deploy
Modo    : 🔌 REAL (USB física)

✅ USB encontrada  (/run/media/udirobotika/A81E-67F1)


## 6.1 Verificar archivos fuente antes de copiar

Parchear obs config

In [40]:
import pathlib, re

obs_path = pathlib.Path(
    f"{DEPLOY_DIR}/policy/release/observation_config.yaml"
)
c = obs_path.read_text()

# Deshabilitar root_z — encoder Oct-9 espera 1751 dims (root_z disabled)
c = re.sub(
    r'(- name: "motion_root_z_position_10frame_step5"\s+enabled:) true',
    r'\1 false', c
)
c = re.sub(
    r'(- name: "motion_root_z_position"\s+enabled:) true',
    r'\1 false', c
)
obs_path.write_text(c)

# Verificar
hits = re.findall(r'motion_root_z_position[^\n]*\n\s+enabled: \w+', c)
for h in hits:
    print(h.strip())
print("\n✅ obs config parcheado — root_z disabled (1751-dim)")

motion_root_z_position_10frame_step5"
      enabled: false
motion_root_z_position"
      enabled: false

✅ obs config parcheado — root_z disabled (1751-dim)


Generando planner

In [43]:
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import numpy as np
import os

def decompose_layer_norm(graph):
    new_nodes = []
    new_inits = []
    for node in graph.node:
        if node.op_type != 'LayerNormalization':
            new_nodes.append(node)
            continue
        x_in    = node.input[0]
        scale_in = node.input[1]
        bias_in  = node.input[2] if len(node.input) > 2 and node.input[2] else None
        y_out    = node.output[0]
        axis, epsilon = -1, 1e-5
        for a in node.attribute:
            if a.name == 'axis':    axis    = a.i
            elif a.name == 'epsilon': epsilon = a.f
        p = y_out + "__ln__"
        pow2 = p + "two";  eps_n = p + "eps"
        new_inits += [
            numpy_helper.from_array(np.float32(2.0),       name=pow2),
            numpy_helper.from_array(np.float32(epsilon),   name=eps_n),
        ]
        mean_o    = p+"mean";   diff_o   = p+"diff"
        diffsq_o  = p+"dsq";   var_o    = p+"var"
        vareps_o  = p+"veps";  std_o    = p+"std"
        norm_o    = p+"norm"
        new_nodes += [
            helper.make_node('ReduceMean', [x_in],          [mean_o],   axes=[axis], keepdims=1),
            helper.make_node('Sub',        [x_in, mean_o],  [diff_o]),
            helper.make_node('Pow',        [diff_o, pow2],  [diffsq_o]),
            helper.make_node('ReduceMean', [diffsq_o],      [var_o],    axes=[axis], keepdims=1),
            helper.make_node('Add',        [var_o, eps_n],  [vareps_o]),
            helper.make_node('Sqrt',       [vareps_o],      [std_o]),
            helper.make_node('Div',        [diff_o, std_o], [norm_o]),
        ]
        if bias_in:
            sc_o = p+"scaled"
            new_nodes += [
                helper.make_node('Mul', [norm_o, scale_in], [sc_o]),
                helper.make_node('Add', [sc_o,   bias_in],  [y_out]),
            ]
        else:
            new_nodes.append(helper.make_node('Mul', [norm_o, scale_in], [y_out]))
    del graph.node[:];          graph.node.extend(new_nodes)
    graph.initializer.extend(new_inits)

MODEL_IN  = (f"{BASE_BLEA}/gear_sonic_deploy/planner/target_vel/V2/planner_sonic.onnx")
MODEL_OUT = (f"{BASE_BLEA}/gear_sonic_deploy/planner/target_vel/V2/planner_sonic_opset14_cf2.onnx")

model = onnx.load(MODEL_IN)
ln_before = sum(1 for n in model.graph.node if n.op_type == 'LayerNormalization')
print(f"LayerNorm nodes antes : {ln_before}")

decompose_layer_norm(model.graph)
del model.opset_import[:]
model.opset_import.append(helper.make_opsetid('', 14))

ln_after = sum(1 for n in model.graph.node if n.op_type == 'LayerNormalization')
print(f"LayerNorm nodes después: {ln_after}")

onnx.save(model, MODEL_OUT)
size_mb = os.path.getsize(MODEL_OUT) / 1e6
print(f"✅ Guardado: {MODEL_OUT}  ({size_mb:.0f} MB)")

LayerNorm nodes antes : 44
LayerNorm nodes después: 0
✅ Guardado: /home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/planner/target_vel/V2/planner_sonic_opset14_cf2.onnx  (774 MB)


In [44]:
FUENTES = {
    "encoder ONNX":   f"{DEPLOY_DIR}/policy/{SKILL_NAME}/model_step_{STEP}_encoder.onnx",
    "decoder ONNX":   f"{DEPLOY_DIR}/policy/{SKILL_NAME}/model_step_{STEP}_decoder.onnx",
    "planner ONNX":   f"{DEPLOY_DIR}/planner/target_vel/V2/planner_sonic_opset14_cf2.onnx",
    "obs config":     f"{DEPLOY_DIR}/policy/release/observation_config.yaml",
    "CMakeLists.txt": f"{DEPLOY_DIR}/CMakeLists.txt",
    "reference dir":  f"{DEPLOY_DIR}/reference/{SKILL_NAME}",
    "src dir":        f"{DEPLOY_DIR}/src",
    "cmake dir":      f"{DEPLOY_DIR}/cmake",
}

print("Verificando fuentes:\n")
all_ok = True
for label, path in FUENTES.items():
    p = pathlib.Path(path)
    ok = p.exists()
    if ok and p.is_file():
        size = f"{p.stat().st_size / 1e6:.1f} MB"
    elif ok and p.is_dir():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        size = f"{n} archivos"
    else:
        size = "NO ENCONTRADO"
    print(f"  {'✅' if ok else '❌'}  {label:<20} {size}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("✅ Todas las fuentes verificadas — proceder a §6.2")
else:
    raise FileNotFoundError("❌ Resolver los ❌ antes de continuar")

Verificando fuentes:

  ✅  encoder ONNX         50.2 MB
  ✅  decoder ONNX         40.9 MB
  ✅  planner ONNX         773.8 MB
  ✅  obs config           0.0 MB
  ✅  CMakeLists.txt       0.0 MB
  ✅  reference dir        9 archivos
  ✅  src dir              62 archivos
  ✅  cmake dir            2 archivos

✅ Todas las fuentes verificadas — proceder a §6.2


## 6.2 Copiar a USB

Copia archivos individuales y directorios completos.
`shutil.copytree` con `dirs_exist_ok=True` es idempotente — se puede
relanzar sin borrar lo anterior.

In [45]:
import shutil, pathlib

def copiar(src, dst_base, label):
    src = pathlib.Path(src)
    dst = pathlib.Path(dst_base) / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        n = sum(1 for _ in dst.rglob("*") if _.is_file())
        print(f"  ✅ {label:<22} → {dst}  ({n} archivos)")
    else:
        shutil.copy2(src, dst)
        print(f"  ✅ {label:<22} → {dst}  ({dst.stat().st_size/1e6:.1f} MB)")

print("Copiando a USB...\n")

BASE_USB_POLICY = f"{USB_DEPLOY}/policy/{SKILL_NAME}"
BASE_USB_ROOT   = USB_DEPLOY

copiar(f"{DEPLOY_DIR}/policy/{SKILL_NAME}/model_step_{STEP}_encoder.onnx", BASE_USB_POLICY, "encoder")
copiar(f"{DEPLOY_DIR}/policy/{SKILL_NAME}/model_step_{STEP}_decoder.onnx", BASE_USB_POLICY, "decoder")
copiar(f"{DEPLOY_DIR}/planner/target_vel/V2/planner_sonic.onnx",            f"{USB_DEPLOY}/planner/target_vel/V2", "planner")
copiar(f"{DEPLOY_DIR}/policy/release/observation_config.yaml",               f"{USB_DEPLOY}/policy/release", "obs config")
copiar(f"{DEPLOY_DIR}/reference/{SKILL_NAME}",                               f"{USB_DEPLOY}/reference", "reference dir")
copiar(f"{DEPLOY_DIR}/src",                                                  BASE_USB_ROOT, "src dir")
copiar(f"{DEPLOY_DIR}/cmake",                                                BASE_USB_ROOT, "cmake dir")
copiar(f"{DEPLOY_DIR}/CMakeLists.txt",                                       BASE_USB_ROOT, "CMakeLists.txt")

print("\n✅ Copia completada")

Copiando a USB...

  ✅ encoder                → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/policy/macarena2/model_step_001000_encoder.onnx  (50.2 MB)
  ✅ decoder                → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/policy/macarena2/model_step_001000_decoder.onnx  (40.9 MB)
  ✅ planner                → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/planner/target_vel/V2/planner_sonic.onnx  (774.0 MB)
  ✅ obs config             → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/policy/release/observation_config.yaml  (0.0 MB)
  ✅ reference dir          → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/reference/macarena2  (9 archivos)
  ✅ src dir                → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/src  (62 archivos)
  ✅ cmake dir              → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/cmake  (2 archivos)
  ✅ CMakeLists.txt         → /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/CMakeLists.txt  (0.0 MB)

✅ Copia completada


## 6.3 Gate §6

In [46]:
ESPERADOS_USB = {
    f"policy/{SKILL_NAME}/model_step_{STEP}_encoder.onnx": 10e6,
    f"policy/{SKILL_NAME}/model_step_{STEP}_decoder.onnx": 10e6,
    f"planner/target_vel/V2/planner_sonic.onnx":           1e3,
    f"policy/release/observation_config.yaml":              1,
    f"CMakeLists.txt":                                      1,
}
ESPERADOS_DIRS_USB = [
    f"reference/{SKILL_NAME}",
    "src",
    "cmake",
]

print("=" * 55)
print("GATE §6 — CHECKLIST")
print("=" * 55)

aprobado = True
base = pathlib.Path(USB_DEPLOY)

for rel, min_bytes in ESPERADOS_USB.items():
    p = base / rel
    ok = p.exists() and p.stat().st_size >= min_bytes
    print(f"  {'✅' if ok else '❌'}  {rel}")
    if not ok:
        aprobado = False

for rel in ESPERADOS_DIRS_USB:
    p = base / rel
    ok = p.is_dir() and any(p.iterdir())
    print(f"  {'✅' if ok else '❌'}  {rel}/")
    if not ok:
        aprobado = False

print()
if aprobado:
    print("✅ Gate §6 CERRADO — USB lista para el Jetson")
    print(f"\n   USB mount : {USB_MOUNT}")
    print(f"   Estructura: {USB_DEPLOY}/")
else:
    print("❌ Gate §6 ABIERTO — revisar errores arriba")

GATE §6 — CHECKLIST
  ✅  policy/macarena2/model_step_001000_encoder.onnx
  ✅  policy/macarena2/model_step_001000_decoder.onnx
  ✅  planner/target_vel/V2/planner_sonic.onnx
  ✅  policy/release/observation_config.yaml
  ✅  CMakeLists.txt
  ✅  reference/macarena2/
  ✅  src/
  ✅  cmake/

✅ Gate §6 CERRADO — USB lista para el Jetson

   USB mount : /run/media/udirobotika/A81E-67F1
   Estructura: /run/media/udirobotika/A81E-67F1/gear_sonic_deploy/


---

ver el onnx

In [48]:
import onnxruntime as ort

ENC_PATH = f"{DEPLOY_DIR}/policy/macarena2/model_step_{int(STEP):06d}_encoder.onnx"
DEC_PATH = f"{DEPLOY_DIR}/policy/macarena2/model_step_{int(STEP):06d}_decoder.onnx"

sess_enc = ort.InferenceSession(ENC_PATH, providers=["CPUExecutionProvider"])
sess_dec = ort.InferenceSession(DEC_PATH, providers=["CPUExecutionProvider"])

print("=== ENCODER ===")
for inp in sess_enc.get_inputs():
    print(f"  IN   {inp.name:<45} {inp.shape}")
for out in sess_enc.get_outputs():
    print(f"  OUT  {out.name:<45} {out.shape}")

print("\n=== DECODER ===")
for inp in sess_dec.get_inputs():
    print(f"  IN   {inp.name:<45} {inp.shape}")
for out in sess_dec.get_outputs():
    print(f"  OUT  {out.name:<45} {out.shape}")

=== ENCODER ===
  IN   obs_dict                                      [1, 1751]
  OUT  encoded_tokens                                [1, 64]

=== DECODER ===
  IN   obs_dict                                      [1, 994]
  OUT  action                                        [1, 29]


In [51]:
import yaml, pathlib

obs_path = pathlib.Path(f"{DEPLOY_DIR}/policy/release/observation_config.yaml")
cfg = yaml.safe_load(obs_path.read_text())

obs = cfg["observations"]
print(f"type(observations): {type(obs)}")
print(f"keys/len: {list(obs.keys()) if isinstance(obs, dict) else len(obs)}")
print()
# Mostrar los primeros 5 elementos para entender la estructura
import json
print(json.dumps(obs if isinstance(obs, dict) else obs[:5], indent=2, default=str))

type(observations): <class 'list'>
keys/len: 6

[
  {
    "name": "token_state",
    "enabled": true
  },
  {
    "name": "his_base_angular_velocity_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_body_joint_positions_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_body_joint_velocities_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_last_actions_10frame_step1",
    "enabled": true
  }
]


In [52]:
import yaml, pathlib, json

obs_path = pathlib.Path(f"{DEPLOY_DIR}/policy/release/observation_config.yaml")
cfg = yaml.safe_load(obs_path.read_text())

print("=== observations (todos) ===")
print(json.dumps(cfg["observations"], indent=2, default=str))

print("\n=== encoder ===")
print(json.dumps(cfg["encoder"], indent=2, default=str))

=== observations (todos) ===
[
  {
    "name": "token_state",
    "enabled": true
  },
  {
    "name": "his_base_angular_velocity_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_body_joint_positions_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_body_joint_velocities_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_last_actions_10frame_step1",
    "enabled": true
  },
  {
    "name": "his_gravity_dir_10frame_step1",
    "enabled": true
  }
]

=== encoder ===
{
  "dimension": 64,
  "use_fp16": false,
  "encoder_observations": [
    {
      "name": "encoder_mode_4",
      "enabled": true
    },
    {
      "name": "motion_joint_positions_10frame_step5",
      "enabled": true
    },
    {
      "name": "motion_joint_velocities_10frame_step5",
      "enabled": true
    },
    {
      "name": "motion_root_z_position_10frame_step5",
      "enabled": false
    },
    {
      "name": "motion_root_z_position",
      "enabled": false
    },
    {
      "

---

# Sección 7 — Jetson: montar USB y organizar

**Precondición:** Gate §6 cerrado. G1 disponible y encendido. USB física
conectada al Jetson Orin NX (puerto USB-A del chasis).

Todos los comandos de §7 corren en **terminal SSH al Jetson** desde el Alienware:

In [ ]:
ssh unitree@192.168.123.164   # pw: 123

⚠️ §7–§12 no se ejecutan desde el notebook — son pasos físicos en el Jetson.
Las celdas siguientes son bloques de referencia para copiar en terminal.

## 7.1 Montar USB en el Jetson



In [ ]:
# Identificar el dispositivo USB
lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT | grep -v loop

ls /media/unitree/A81E-67F1/gear_sonic_deploy/

Salida esperada:

In [ ]:
CMakeLists.txt cmake/ planner/ policy/ reference/ src/

## 7.2 Copiar gear_sonic_deploy al Jetson



In [ ]:
USB=/media/unitree/A81E-67F1/gear_sonic_deploy
DEST=~/blea/gear_sonic_deploy

# Política + planner
cp -rv $USB/policy/  $DEST/policy/
cp -rv $USB/planner/ $DEST/planner/
cp -rv $USB/reference/ $DEST/reference/

# Verificar los 3 ONNX críticos
ls -lh $DEST/policy/macarena2/
ls -lh $DEST/planner/target_vel/V2/
ls -lh $DEST/policy/release/observation_config.yaml

## 7.3 Gate §7

In [ ]:
# Verificar los archivos críticos
python3 - << 'EOF'
import pathlib

BASE = pathlib.Path.home() / "blea/gear_sonic_deploy"
SKILL = "macarena2"
STEP  = "001000"

checks = [
    BASE / f"policy/{SKILL}/model_step_{STEP}_encoder.onnx",
    BASE / f"policy/{SKILL}/model_step_{STEP}_decoder.onnx",
    BASE / f"planner/target_vel/V2/planner_sonic.onnx",
    BASE / f"policy/release/observation_config.yaml",
    BASE / f"reference/{SKILL}",
    BASE / "src",
    BASE / "CMakeLists.txt",
]

all_ok = True
print("GATE §7 — CHECKLIST")
print("=" * 45)
for p in checks:
    ok = p.exists()
    size = f"{p.stat().st_size/1e6:.1f} MB" if ok and p.is_file() else \
           f"{sum(1 for _ in p.rglob('*') if _.is_file())} archivos" if ok else "NO ENCONTRADO"
    print(f"  {'✅' if ok else '❌'}  {p.name}  ({size})")
    if not ok:
        all_ok = False

print()
print("✅ Gate §7 CERRADO — proceder a §8" if all_ok else "❌ Gate §7 ABIERTO")
EOF

---

# §8 — Deploy en G1 Jetson






### §8.0 Pre-check: artefactos en G1

Verificar que la USB está montada y los artefactos llegaron íntegros.

**En el G1 (SSH desde Alienware):**

In [ ]:
ssh unitree@192.168.123.164
ls /mnt/usb_blea/gear_sonic_deploy/

> Si el directorio no existe o está vacío → volver a §7 y repetir transferencia USB.

### §8.1 Bootstrap de red: NAT Alienware → G1

El G1 Jetson no tiene salida a internet. El Alienware actúa como gateway NAT
enrutando el tráfico del G1 por su interfaz WiFi.

**Interfaces relevantes (Alienware m18 R2):**
| Interfaz | Nombre | IP |
|---|---|---|
| WiFi (internet) | `wlp109s0f0` | DHCP (10.16.x.x) |
| Ethernet (robot) | `enp110s0` | `192.168.123.105` |
| G1 Jetson | — | `192.168.123.164` |

**En el Alienware:**



In [ ]:
sudo sysctl -w net.ipv4.ip_forward=1
sudo iptables -t nat -A POSTROUTING -o wlp109s0f0 -j MASQUERADE
sudo iptables -A FORWARD -i enp110s0 -o wlp109s0f0 -j ACCEPT
sudo iptables -A FORWARD -i wlp109s0f0 -o enp110s0 -m state --state RELATED,ESTABLISHED -j ACCEPT

**En el G1:**




In [ ]:
sudo ip route add default via 192.168.123.105
echo "nameserver 8.8.8.8" | sudo tee /etc/resolv.conf

**Verificación:**




In [ ]:
ping -c 2 google.com   # debe responder con 0% packet loss

> ⚠️ Esta configuración es **temporal** — desaparece al reiniciar ambas máquinas.
> Repetir este paso al inicio de cada sesión de deploy si el G1 necesita acceso a internet.

### §8.2 Bootstrap de deps: apt + ORT aarch64

**Contexto del Jetson:**
- Ubuntu 20.04 focal (arm64)
- JetPack 5.1.1 (R35.3.1)
- CUDA 11.4, TensorRT 8.5.2
- Mirror apt: mirrors.ustc.edu.cn (cert falla si RTC = 1970)

**Fix de reloj antes de apt (obligatorio si RTC no sincronizó):**





In [ ]:
sudo date -s "YYYY-MM-DD HH:MM:SS"   # fecha real del momento
date                                   # verificar

**Deps apt:**

In [ ]:
sudo apt-get install -y libmsgpack-dev nlohmann-json3-dev

**ONNX Runtime para aarch64 + JetPack 5.1.1:**

In [ ]:
# Descargar prebuilt ORT 1.16.3 aarch64 desde GitHub
cd ~
wget https://github.com/microsoft/onnxruntime/releases/download/v1.16.3/onnxruntime-linux-aarch64-1.16.3.tgz
tar -xzf onnxruntime-linux-aarch64-1.16.3.tgz
sudo cp -r onnxruntime-linux-aarch64-1.16.3/include/* /usr/local/include/
sudo cp -r onnxruntime-linux-aarch64-1.16.3/lib/* /usr/local/lib/
sudo ldconfig

# Verificar que CMake puede encontrar el config
find /usr/local -name "onnxruntimeConfig.cmake" 2>/dev/null

> ⚠️ Si el tarball no incluye `onnxruntimeConfig.cmake`, hay que crear el wrapper
> manualmente (ver nota §8.2.1 abajo).

#### §8.2.1 Wrapper CMake para ORT (prebuilt sin config)

El tarball oficial de ORT 1.16.3 aarch64 no incluye `onnxruntimeConfig.cmake`.
Crearlo manualmente:

In [ ]:
sudo mkdir -p /usr/local/lib/cmake/onnxruntime
sudo tee /usr/local/lib/cmake/onnxruntime/onnxruntimeConfig.cmake > /dev/null << 'EOF'
if(TARGET onnxruntime::onnxruntime)
  return()
endif()
find_library(ONNXRUNTIME_LIB onnxruntime PATHS /usr/local/lib NO_DEFAULT_PATH)
find_path(ONNXRUNTIME_INCLUDE_DIR onnxruntime_cxx_api.h PATHS /usr/local/include NO_DEFAULT_PATH)
include(FindPackageHandleStandardArgs)
find_package_handle_standard_args(onnxruntime REQUIRED_VARS ONNXRUNTIME_LIB ONNXRUNTIME_INCLUDE_DIR)
add_library(onnxruntime::onnxruntime SHARED IMPORTED)
set_target_properties(onnxruntime::onnxruntime PROPERTIES
  IMPORTED_LOCATION "${ONNXRUNTIME_LIB}"
  INTERFACE_INCLUDE_DIRECTORIES "${ONNXRUNTIME_INCLUDE_DIR}"
)
EOF

Verificar

In [ ]:
find /usr/local -name "onnxruntimeConfig.cmake"

**Estado final de deps §8.2:**
| Dep | Versión | Fuente |
|---|---|---|
| libeigen3-dev | 3.3.7 | apt (preinstalado) |
| libzmq3-dev | 4.3.2 | apt (preinstalado) |
| unitree_sdk2 | — | preinstalado por Unitree |
| TensorRT | 8.5.2 | JetPack 5.1.1 |
| libmsgpack-dev | 3.0.1 | apt |
| nlohmann-json3-dev | 3.7.3 | apt |
| onnxruntime | 1.16.3 | GitHub prebuilt aarch64 + wrapper CMake manual |

### §8.3 Build del binary en Jetson: cmake + make

**Contexto del entorno Jetson:**
- Ubuntu 20.04 focal arm64 / JetPack 5.1.1 / GCC 9.4.0 (default)
- CUDA 11.4, TensorRT 8.5.2, cppzmq 4.3.1 (sistema)

**Tres incompatibilidades con el código fuente (escrito para TRT 8.6+ / GCC 10+ / cppzmq 4.7+):**
1. GCC 9 no soporta `-std=c++20` ni los headers `<span>` / `<stop_token>` → instalar gcc-10
2. `thirdparty/unitree_sdk2` es un git submodule que no viaja en el tarball USB → parchear CMakeLists.txt
3. `InferenceEngine.cpp` usa tipos TRT 8.6+ (`kINT64`, `kBF16`, `kFP8`, `setShapeValuesV2`) → parchear
4. `zmq::sockopt` requiere cppzmq ≥ 4.7.0 → overlay con 4.9.0

**Binary de salida:** `target/release/g1_deploy_onnx_ref` (relativo a la raíz del repo, NO dentro de `build/`)

cell 1 — instalar gcc-10:

In [ ]:
sudo apt install -y gcc-10 g++-10
gcc-10 --version | head -1

cell 2 — overlay cppzmq 4.9.0:

In [ ]:
wget -q https://raw.githubusercontent.com/zeromq/cppzmq/v4.9.0/zmq.hpp \
     -O /tmp/zmq490.hpp
sudo cp /tmp/zmq490.hpp /usr/include/zmq.hpp
sudo cp /tmp/zmq490.hpp /usr/local/include/zmq.hpp

echo "=== versión instalada ==="
grep "CPPZMQ_VERSION_M" /usr/include/zmq.hpp | head -3

cell 3 — patch CMakeLists.txt (submodule → system install):

In [ ]:
cd ~/blea/gear_sonic_deploy
cp CMakeLists.txt CMakeLists.txt.bak

# 1. add_subdirectory → find_package (unitree_sdk2 instalado en /usr/local)
sed -i 's|add_subdirectory(thirdparty/unitree_sdk2)|# add_subdirectory(thirdparty/unitree_sdk2)  # patched: system install\nfind_package(unitree_sdk2 REQUIRED)|' CMakeLists.txt

# 2. include path al path de sistema
sed -i 's|INTERFACE ${CMAKE_CURRENT_SOURCE_DIR}/thirdparty/unitree_sdk2/include|INTERFACE /usr/local/include|' CMakeLists.txt

echo "=== verificar ==="
grep -n "add_subdirectory(thirdparty\|find_package(unitree_sdk2" CMakeLists.txt

cell 4 — patch InferenceEngine.cpp (TRT 8.5 compat):

In [ ]:
FILE=~/blea/gear_sonic_deploy/src/TRTInference/InferenceEngine.cpp
cp $FILE $FILE.bak

# Fix 1: comentar DataTypes inexistentes en TRT 8.5
sed -i \
  -e 's/case nvinfer1::DataType::kINT64:/\/\/ [TRT8.5] case nvinfer1::DataType::kINT64:/' \
  -e 's/case nvinfer1::DataType::kBF16:/\/\/ [TRT8.5] case nvinfer1::DataType::kBF16:/' \
  -e 's/case nvinfer1::DataType::kFP8:/\/\/ [TRT8.5] case nvinfer1::DataType::kFP8:/' \
  $FILE

# Fix 2: setShapeValuesV2 → setShapeValues (renombrado en TRT 8.6)
sed -i 's/setShapeValuesV2/setShapeValues/g' $FILE

# Fix 3: convertir vector<long> → vector<int32_t> para setShapeValues
python3 << 'EOF'
FILE = "/home/unitree/blea/gear_sonic_deploy/src/TRTInference/InferenceEngine.cpp"
with open(FILE, 'r') as f:
    lines = f.readlines()
out = []
inserted = False
for line in lines:
    if 'min_v.data()' in line and 'setShapeValues' in line and not inserted:
        indent = line[:len(line) - len(line.lstrip())]
        out.append(indent + 'std::vector<int32_t> min_v32(min_v.begin(), min_v.end());\n')
        out.append(indent + 'std::vector<int32_t> opt_v32(opt_v.begin(), opt_v.end());\n')
        out.append(indent + 'std::vector<int32_t> max_v32(max_v.begin(), max_v.end());\n')
        out.append(line.replace('min_v.data()', 'min_v32.data()')
                       .replace('(int32_t)min_v.size()', '(int32_t)min_v32.size()'))
        inserted = True
    elif 'opt_v.data()' in line and 'setShapeValues' in line:
        out.append(line.replace('opt_v.data()', 'opt_v32.data()')
                       .replace('(int32_t)opt_v.size()', '(int32_t)opt_v32.size()'))
    elif 'max_v.data()' in line and 'setShapeValues' in line:
        out.append(line.replace('max_v.data()', 'max_v32.data()')
                       .replace('(int32_t)max_v.size()', '(int32_t)max_v32.size()'))
    else:
        out.append(line)
with open(FILE, 'w') as f:
    f.writelines(out)
print("patch OK")
EOF

cell 5 — cmake configure:

In [ ]:
cd ~/blea/gear_sonic_deploy
rm -rf build && mkdir build && cd build

cmake .. \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_PREFIX_PATH="/usr/local" \
  -DCMAKE_CXX_COMPILER=g++-10 \
  -DCMAKE_C_COMPILER=gcc-10 \
  2>&1 | tee cmake_config.log

RESULT=${PIPESTATUS[0]}
echo "--- cmake exit: $RESULT ---"
grep -E "(Error|Configuring done)" cmake_config.log | head -5

cell 6 — make (solo target de deploy):

In [ ]:
cd ~/blea/gear_sonic_deploy/build
make -j$(nproc) g1_deploy_onnx_ref 2>&1 | tee make.log
RESULT=${PIPESTATUS[0]}
echo "--- make exit: $RESULT ---"
tail -4 make.log

cell 7 — gate §8.3:

In [ ]:
BIN=~/blea/gear_sonic_deploy/target/release/g1_deploy_onnx_ref

if [ ! -f "$BIN" ]; then
  echo "❌ GATE FAIL — g1_deploy_onnx_ref no encontrado"
else
  echo "✅ GATE PASS"
  ls -lh "$BIN"
  file "$BIN"
fi

### §8.4.2 — TRT Engine Build: Smoke Test

In [ ]:
sin fix

In [ ]:
cd ~/blea/gear_sonic_deploy

./target/release/g1_deploy_onnx_ref lo \
    policy/macarena2/model_step_001000_decoder.onnx \
    reference/macarena2/ \
    --encoder-file policy/macarena2/model_step_001000_encoder.onnx \
    --planner-file planner/target_vel/V2/planner_sonic_opset14_cf2.onnx \
    --obs-config policy/release/observation_config.yaml \
    --input-type zmq \
    --disable-crc-check 2>&1 | tee trt_build.log &

sleep 90
kill %1 2>/dev/null

grep -E "(Successfully converted|Dimension match|LowState|Error|error)" trt_build.log | head -20
echo "--- §8.4.2 ---"


Lanza el binario en modo `lo` (loopback, sin G1 físico) para verificar que los tres engines TRT compilan e inicializan correctamente.

El planner `planner_sonic.onnx` (opset 17) requirió una cadena de tres fixes antes de que TRT 8.5 lo aceptara:

| Fix | Problema | Solución |
|-----|----------|----------|
| 1 | `TopK`: K input dinámico (`/Gather_139_output_0`) — TRT requiere initializer | Polygraphy constant folding → `_cf.onnx` (8284→3750 nodos) |
| 2 | `Einsum` con ellipsis (`... x n m, ... x n o -> ... x m o`) — TRT 8.5 no soporta `...` | Reemplazar con `Transpose(perm=[0,1,2,4,3])` + `MatMul` → `_cf2.onnx` |
| 3 | Encoder dim mismatch: `got 1762, expected 1751` (delta=11) | `motion_root_z_position_10frame_step5` (+10) y `motion_root_z_position` (+1) habilitadas en `observation_config.yaml` pero ausentes en todos los `encoder_modes` → `enabled: false` |

**Deps Jetson requeridos (instalar una vez):**


In [ ]:
pip install onnxruntime
pip install "onnx-graphsurgeon>=0.3.27" --extra-index-url https://pypi.ngc.nvidia.com

Fix 1: Polygraphy constant folding

In [ ]:
cd ~/blea/gear_sonic_deploy

pip install onnxruntime \
    "onnx-graphsurgeon>=0.3.27" \
    --extra-index-url https://pypi.ngc.nvidia.com -q

polygraphy surgeon sanitize \
    --fold-constants \
    planner/target_vel/V2/planner_sonic_opset14.onnx \
    -o planner/target_vel/V2/planner_sonic_opset14_cf.onnx

python3 - <<'EOF'
import onnx
m = onnx.load("planner/target_vel/V2/planner_sonic_opset14_cf.onnx")
inits = {i.name for i in m.graph.initializer}
bad = [n for n in m.graph.node
       if n.op_type == "TopK" and n.input[1] not in inits]
print(f"Nodes: {len(m.graph.node)} | Initializers: {len(inits)}")
print(f"Dynamic-K TopK restantes: {len(bad)}  <- esperado: 0")
assert len(bad) == 0, "TopK K sigue dinamico -- verificar onnx-graphsurgeon>=0.3.27"
print("OK  Fix 1 -- planner_sonic_opset14_cf.onnx listo")
EOF

Fix 2: Einsum ellipsis patch

In [ ]:
cd ~/blea/gear_sonic_deploy

python3 - <<'EOF'
import onnx
from onnx import TensorProto

IN  = "planner/target_vel/V2/planner_sonic_opset14_cf.onnx"
OUT = "planner/target_vel/V2/planner_sonic_opset14_cf2.onnx"

m = onnx.load(IN)
graph = m.graph

# Localizar nodo Einsum con ellipsis
einsum_node = None
einsum_idx  = None
for i, node in enumerate(graph.node):
    if node.op_type == "Einsum":
        for a in node.attribute:
            if a.name == "equation" and b"..." in a.s:
                einsum_node = node
                einsum_idx  = i
                break
    if einsum_node:
        break

assert einsum_node is not None, "Einsum ellipsis node not found"
print(f"Einsum encontrado en indice {einsum_idx}: {einsum_node.name!r}")
# equation: '... x n m, ... x n o -> ... x m o'
# shapes:   [1,64,34,3,3] x2  ->  [1,64,34,3,3]
# fix:      Transpose(A, perm=[0,1,2,4,3]) @ B

inp_a        = einsum_node.input[0]
inp_b        = einsum_node.input[1]
out          = einsum_node.output[0]
transposed_a = "/Einsum_transposed_A_patch"

transpose_node = onnx.helper.make_node(
    "Transpose",
    inputs=[inp_a],
    outputs=[transposed_a],
    name="/Einsum_Transpose_patch",
    perm=[0, 1, 2, 4, 3],
)
matmul_node = onnx.helper.make_node(
    "MatMul",
    inputs=[transposed_a, inp_b],
    outputs=[out],
    name="/Einsum_MatMul_patch",
)
vi = onnx.helper.make_tensor_value_info(
    transposed_a, TensorProto.FLOAT, [1, 64, 34, 3, 3]
)
graph.value_info.append(vi)

new_nodes = []
for i, node in enumerate(graph.node):
    if i == einsum_idx:
        new_nodes.append(transpose_node)
        new_nodes.append(matmul_node)
    else:
        new_nodes.append(node)
del graph.node[:]
graph.node.extend(new_nodes)

onnx.save(m, OUT)
print(f"Guardado: {OUT}")

# Verificar
m2    = onnx.load(OUT)
inits = {i.name for i in m2.graph.initializer}
bad_einsum = [n for n in m2.graph.node
              if n.op_type == "Einsum"
              and any(b"..." in a.s for a in n.attribute if a.name == "equation")]
bad_topk   = [n for n in m2.graph.node
              if n.op_type == "TopK" and n.input[1] not in inits]
print(f"Ellipsis Einsum restantes : {len(bad_einsum)}  <- esperado: 0")
print(f"Dynamic-K TopK restantes  : {len(bad_topk)}   <- esperado: 0")
assert len(bad_einsum) == 0 and len(bad_topk) == 0, "Fix incompleto"
print("OK  Fix 2 -- planner_sonic_opset14_cf2.onnx listo")
EOF

Fix 3: obs config + smoke test

In [ ]:
%%bash
cd ~/blea/gear_sonic_deploy

# Fix 3 — deshabilitar obs ausentes en todos los encoder_modes (delta -11 dims)
cp policy/release/observation_config.yaml \
   policy/release/observation_config.yaml.bak

python3 - <<'EOF'
path = "policy/release/observation_config.yaml"
content = open(path).read()
content = content.replace(
    '- name: "motion_root_z_position_10frame_step5"\n      enabled: true',
    '- name: "motion_root_z_position_10frame_step5"\n      enabled: false'
)
content = content.replace(
    '- name: "motion_root_z_position"\n      enabled: true',
    '- name: "motion_root_z_position"\n      enabled: false'
)
open(path, "w").write(content)
import subprocess
out = subprocess.check_output(
    ["grep", "-A1", "motion_root_z_position", path], text=True
)
print(out)
assert "enabled: false" in out, "Patch no aplicado"
print("OK  Fix 3 -- observation_config.yaml parcheado (-11 dims)")
EOF

# Smoke test — modo loopback (sin G1 fisico)
# Gate de exito: "LowState is not available, waiting for robot to be ready"
# Eso indica que los 3 engines TRT inicializaron y el binario espera DDS
./target/release/g1_deploy_onnx_ref lo \
    policy/saludo_militar/model_step_001000_decoder.onnx \
    reference/saludo_militar/ \
    --encoder-file policy/saludo_militar/model_step_001000_encoder.onnx \
    --planner-file planner/target_vel/V2/planner_sonic_opset14_cf2.onnx \
    --obs-config policy/release/observation_config.yaml \
    --input-type zmq \
    --disable-crc-check 2>&1 | tee trt_build.log &

sleep 60
kill %1 2>/dev/null

grep -E "(Successfully converted|Dimension match|LowState)" trt_build.log
echo "--- §8.4.2 CERRADO ---"

---

# §9 — Deploy Real en G1

> **Pre-requisitos físicos antes de ejecutar cualquier celda:**
> - G1 encendido y en posición de damping (no en pie libre)
> - Cable ethernet Jetson ↔ G1 conectado
> - `CYCLONEDDS_URI` configurado con la interfaz correcta
> - `DDS_DOMAIN_ID` coincide entre Jetson y G1 (default: 1)

### Sub-secciones

| § | Descripción | Gate |
|---|-------------|------|
| §9.1 | Verificar conectividad red Jetson ↔ G1 | `ping` responde en <5ms |
| §9.2 | Configurar CycloneDDS y verificar DDS | `LowState` recibido vía DDS listener |
| §9.3 | Lanzar deploy real (interfaz `eth0`) | Binary arranca, `LowState is available` |
| §9.4 | Primera inferencia — saludo militar | Joints se mueven según `reference/saludo_militar/` |

> **Nota:** a diferencia del smoke test (§8.4.2 con `lo`),
> aquí el binario recibe `LowState` real del G1 y envía `LowCmd` de vuelta.
> Tener a alguien sosteniendo el robot o usar el arnés en la primera ejecución.

## §9.1 — Verificar red Jetson ↔ G1

G1 expone IP fija en la interfaz ethernet interna

In [ ]:
cd ~/blea/gear_sonic_deploy

python3 - <<'EOF'
import subprocess, re
G1_IP = "192.168.123.161"
r = subprocess.run(["ping", "-c", "4", "-W", "1", G1_IP], capture_output=True, text=True)
print(r.stdout)
assert "0% packet loss" in r.stdout, "FAIL  G1 no responde"
m = re.search(r'rtt.*=\s*([\d.]+)/', r.stdout)
if m:
    rtt = float(m.group(1))
    assert rtt < 5.0, f"FAIL  RTT {rtt}ms > 5ms"
print("OK  §9.1 cerrado")
EOF

## §9.2 — Verificar DDS (LowState disponible)


In [ ]:
cd ~/blea/gear_sonic_deploy

./target/release/g1_deploy_onnx_ref eth0 \
    policy/macarena2/model_step_001000_decoder.onnx \
    reference/macarena2/ \
    --encoder-file policy/macarena2/model_step_001000_encoder.onnx \
    --planner-file planner/target_vel/V2/planner_sonic_opset14_cf2.onnx \
    --obs-config policy/release/observation_config.yaml \
    --input-type zmq \
    --disable-crc-check </dev/null 2>&1 | tee deploy_dds_check.log &
PID=$!
sleep 30
kill $PID 2>/dev/null
wait

python3 - <<'EOF'
log = open("deploy_dds_check.log").read()
assert "Init Done" in log, "FAIL  Init Done no encontrado\n" + log[-500:]
print("OK  §9.2 cerrado — DDS vivo, LowState recibido")
EOF

## §9.3 — Deploy 

In [ ]:
cd ~/blea/gear_sonic_deploy && ./target/release/g1_deploy_onnx_ref eth0 \
    policy/macarena2/model_step_001000_decoder.onnx \
    reference/macarena2/ \
    --encoder-file policy/macarena2/model_step_001000_encoder.onnx \
    --planner-file planner/target_vel/V2/planner_sonic_opset14_cf2.onnx \
    --obs-config policy/release/observation_config.yaml \
    --disable-crc-check